# Extract CellViT nucleus features

Preflight runs first: one real batch through the exact checkpoint, postprocessor,
DINO crop encoder and cache writer, plus a runtime estimate. Full extraction
starts only if that passes, so a bad path or scale does not burn a session.

**Both T4s are used.** The patch range is split into contiguous blocks of *whole
batches*, one worker process per GPU, each writing its own per-batch `.npz` files
into the shared `.build` directory. A final single-process pass merges them into
the cache layout the sampler reads.

Why whole batches, never a split one: `CellViTPatchExtractor._prepare_batch` pads
every image in a batch up to the largest one in that batch, so a batch with
different members is a different forward pass on any variable-size dataset.
Assigning whole batches means each batch has exactly the members it would have
had in a one-GPU run.

**Seeding.** Each shard process passes the same `--seed`, and `get_data_loaders`
derives the train/test split from it. Two workers with different seeds would
shard two different splits. The cache directory name carries the seed, and the
manifest carries a fingerprint of the sample order, which the sampler verifies
against its own split before using the cache.

**Resume.** A batch file that already exists is not recomputed, so a session
that dies part-way — or hits the 12-hour limit — can simply be re-run.

Needs `requirements-cellvit.txt` (numba / cv2 / scikit-image). No other notebook
does.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-cellvit.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
# ---- EDIT ONLY THIS CELL ----
DATASET = "pathmnist"          # pathmnist | skintissue; HistoSet needs per-source MPP
SEED = 42
DATA_PATH = DATA_PATHS[DATASET]

CHECKPOINT_CANDIDATES = [
    DATA_ROOT / "CellViT-256-x40-AMP.pth",
    Path("/kaggle/input/cellvit-checkpoints/CellViT-256-x40-AMP.pth"),
]
CHECKPOINT_PATH = str(
    next((p for p in CHECKPOINT_CANDIDATES if p.is_file()), CHECKPOINT_CANDIDATES[0])
)

# PathMNIST source pixels are 0.5 MPP. MODEL_MPP/MAGNIFICATION must match the checkpoint.
INPUT_MPP = 0.5
MODEL_MPP = 0.25
MAGNIFICATION = 40

CACHE_DIR = "/kaggle/working/cellvit_features"
DINO_MODEL = "facebook/dinov2-base"   # or a mounted local model directory
BATCH_SIZE = 2
DINO_CROP_BATCH_SIZE = 32
SMOKE_SAMPLES = 8                     # spread across the train set for the estimate
MAX_ESTIMATED_HOURS = 10.0            # fail before wasting a Kaggle session
MAX_CELLS_PER_PATCH = 16              # T4-safe; keep identical across every variant
OVERWRITE = False

# Use both T4s. False forces one GPU, which is the reference path.
PARALLEL = True

assert Path(DATA_PATH).exists(), DATA_PATH
assert Path(CHECKPOINT_PATH).is_file(), CHECKPOINT_PATH
assert MAGNIFICATION in (20, 40)
assert not str(CACHE_DIR).startswith("/kaggle/input"), "CACHE_DIR must be writable"

In [ ]:
from huggingface_hub import snapshot_download

# Pull the DINO weights once in the parent: two shard workers racing to populate
# the same Hugging Face cache is avoidable work and an avoidable failure mode.
if "/" in DINO_MODEL and not Path(DINO_MODEL).exists():
    print(f"Downloading {DINO_MODEL} ...")
    snapshot_download(repo_id=DINO_MODEL)

In [ ]:
# Exact one-batch integration test. Do not continue if this cell fails.
# Runs on a single GPU: it is a correctness gate, not a throughput test, and the
# runtime it estimates is the ONE-GPU figure. With PARALLEL the wall clock is
# roughly half that, but the estimate stays the conservative number to plan by.
preflight = [
    sys.executable, "scripts/preflight_cellvit.py",
    "--dataset", DATASET, "--data_path", DATA_PATH,
    "--checkpoint", CHECKPOINT_PATH, "--cache_dir", CACHE_DIR,
    "--input_mpp", str(INPUT_MPP), "--model_mpp", str(MODEL_MPP),
    "--magnification", str(MAGNIFICATION),
    "--vit_name", DINO_MODEL, "--seed", str(SEED),
    "--smoke_samples", str(SMOKE_SAMPLES),
    "--cellvit_batch_size", str(BATCH_SIZE),
    "--dino_crop_batch_size", str(DINO_CROP_BATCH_SIZE),
    "--max_estimated_hours", str(MAX_ESTIMATED_HOURS),
]
if MAX_CELLS_PER_PATCH is not None:
    preflight += ["--max_cells_per_patch", str(MAX_CELLS_PER_PATCH)]
subprocess.check_call(preflight)

In [ ]:
from scripts.cellvit_shard_worker import build_shard_jobs, run_cellvit_shard
from utils import nucleus_archive_stem, slugify
from utils.parallel import run_variants_parallel, visible_gpu_count

# Every flag both the shard workers and the assembly pass need. Kept in one dict
# so a shard and its assembly cannot disagree about scale or checkpoint — a
# disagreement the .build state check would reject only after the GPU time was
# already spent.
OPTIONS = {
    "dataset": DATASET,
    "data_path": DATA_PATH,
    "checkpoint": CHECKPOINT_PATH,
    "cache_dir": CACHE_DIR,
    "input_mpp": INPUT_MPP,
    "model_mpp": MODEL_MPP,
    "magnification": MAGNIFICATION,
    "vit_name": DINO_MODEL,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "dino_crop_batch_size": DINO_CROP_BATCH_SIZE,
}
if MAX_CELLS_PER_PATCH is not None:
    OPTIONS["max_cells_per_patch"] = MAX_CELLS_PER_PATCH

SHARDS = max(1, visible_gpu_count() if PARALLEL else 1)
print(f"GPUs visible: {visible_gpu_count()} | shards: {SHARDS}")
print("cache:", Path(CACHE_DIR) / f"{DATASET}_seed{SEED}")

In [ ]:
import time

manifest = Path(CACHE_DIR) / f"{DATASET}_seed{SEED}" / "manifest.json"
started = time.time()

if manifest.is_file() and not OVERWRITE:
    print("completed cache already exists; skipping extraction:", manifest)
elif SHARDS == 1:
    run_cellvit_shard(OPTIONS, overwrite=OVERWRITE)
else:
    # Phase 1: every GPU extracts its own block of batches into .build/.
    jobs = build_shard_jobs(OPTIONS, SHARDS, overwrite=OVERWRITE)
    results = run_variants_parallel(jobs, run_cellvit_shard, num_workers=SHARDS)
    failed = [r["label"] for r in results if not r["ok"]]
    assert not failed, f"shards failed: {failed}"

    # Phase 2: one process merges them. This rebuilds the ragged `offsets` array
    # by reading every batch file in patch order, so the result does not depend
    # on which worker wrote which file, and a missing batch is a hard error
    # rather than a silently short cache. No GPU, no models loaded.
    run_cellvit_shard(OPTIONS, assemble_only=True)

print(f"extraction total {time.time() - started:.0f}s")

In [ ]:
# Verify the cache the sampler will actually read: load it through the same
# loader `main.py` uses, and check its sample order against a freshly built
# split. A cache whose rows are misaligned with the split loads fine and
# silently corrupts every experiment downstream, so this check is the point.
import numpy as np

from data.identity import sample_order_fingerprint
from data.loaders import get_data_loaders, get_sample_ids
from features.cellvit.cache import load_cellvit_cache
from utils import set_seed

set_seed(SEED)
train_loader, _, _ = get_data_loaders(DATA_PATH, SEED, verbose=True)
expected_ids = get_sample_ids(train_loader.dataset)

cache = load_cellvit_cache(
    str(Path(CACHE_DIR) / f"{DATASET}_seed{SEED}"),
    expected_sample_ids=expected_ids,   # raises if the order does not match
)
assert cache.num_patches == len(expected_ids), (cache.num_patches, len(expected_ids))
assert cache.manifest["sample_fingerprint"] == sample_order_fingerprint(expected_ids)
assert cache.manifest["dataset"] == DATASET and cache.manifest["seed"] == SEED
assert cache.offsets[0] == 0 and cache.offsets[-1] == cache.num_cells

empty = int(np.sum(np.diff(cache.offsets) == 0))
print(f"OK patches={cache.num_patches} cells={cache.num_cells} "
      f"mean={cache.num_cells / max(1, cache.num_patches):.1f}/patch")
print(f"    cellvit={cache.features('cellvit_embedding').shape} "
      f"crop_dino={cache.features('crop_dino').shape}")
print(f"    patches with no nucleus: {empty} "
      f"({100 * empty / max(1, cache.num_patches):.1f}%) -> scalpel missing_impute")
assert not (Path(CACHE_DIR) / f"{DATASET}_seed{SEED}" / ".build").exists(), (
    "the .build directory survives only when assembly did not finish"
)

In [ ]:
# Archive everything this notebook produced, ready to publish as a Kaggle
# Dataset. The archive is written OUTSIDE the directory being archived: naming
# it inside would include a partial copy of itself on a second run.
#
# The name carries dataset + seed + both encoders + the cell cap, because those
# are exactly the axes that make two nucleus caches non-interchangeable, and the
# sampler reads them back out of the manifest.
import json
import shutil

SOURCE = Path(CACHE_DIR)
ARCHIVE_DIR = Path("/kaggle/working/archive")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
assert ARCHIVE_DIR.resolve() not in [SOURCE.resolve(), *SOURCE.resolve().parents], (
    "the archive directory must not sit inside the directory being archived"
)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

STEM = nucleus_archive_stem(
    DATASET, SEED, Path(CHECKPOINT_PATH).stem, DINO_MODEL, MAX_CELLS_PER_PATCH
)
ARCHIVE = ARCHIVE_DIR / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6
print(f"{ARCHIVE}.zip  ({size_mb:.1f} MB)")

# The zip root holds `<dataset>_seed<seed>/`, which is the directory name the run
# notebook probes for, so extracting the archive anywhere is enough.
print("contents:")
for path in sorted(SOURCE.rglob("*")):
    if path.is_file() and ".build" not in path.parts and "qc" not in path.parts:
        print("  ", path.relative_to(SOURCE), f"{path.stat().st_size / 1e6:.1f} MB")

# slugify appends a digest of the full stem when it has to truncate: an x20 and
# an x40 cache would otherwise collapse onto one slug and version over each other.
SLUG = slugify(STEM)
(ARCHIVE_DIR / "dataset-metadata.json").write_text(json.dumps({
    "title": f"cellvit nucleus features {DATASET} seed{SEED}",
    "id": f"{os.environ.get('KAGGLE_USERNAME', 'YOUR_USERNAME')}/{SLUG}",
    "licenses": [{"name": "CC0-1.0"}],
}, indent=2), encoding="utf-8")

print("\nTo publish (run in a terminal with ~/.kaggle/kaggle.json):")
print(f"  kaggle datasets create   -p {ARCHIVE_DIR} --dir-mode zip   # first time")
print(f"  kaggle datasets version  -p {ARCHIVE_DIR} -m 'update'      # afterwards")
print("\nThen attach it; run_al_sampler.ipynb probes for "
      f"'{DATASET}_seed{SEED}/manifest.json', so no path is hard-coded.")